# Two-Axis Regime Detection Pipeline

Step-by-step walkthrough of the dual-axis regime detection pipeline for ISO New England electricity prices.

**Pipeline overview:**
1. Load data
2. Preprocessing: arcsinh + MSTL decomposition
3. Sliding windows (W=512h, S=6h)
4. Feature Engineering (15 features on stationary increments Δr_t)
5. MOMENT embedding (1024D on persistent levels r_t)
6. Diffusion Maps (dimensionality reduction)
7. ToMATo clustering (topological mode detection)
8. Cross-diagnostic (η² separation analysis)
9. Tukey HSD merge → final regimes
10. Independence test (ARI)

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import warnings, gc, sys, os

from scipy.stats import skew, kurtosis, studentized_range
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from statsmodels.tsa.seasonal import MSTL

warnings.filterwarnings('ignore')

# Navigate to project root
os.chdir(os.path.join(os.path.dirname(os.path.abspath('.')), ''))
sys.path.insert(0, '..')

SEED = 42
W, S = 512, 6
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 1. Load Data

In [ ]:
df_raw = pd.read_parquet('../isone_dataset.parquet')
print(f'Shape: {df_raw.shape}')
print(f'Period: {df_raw["datetime"].min()} to {df_raw["datetime"].max()}')
print(f'\nLMP statistics:')
df_raw['lmp'].describe()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(pd.to_datetime(df_raw['datetime']), df_raw['lmp'], lw=0.3, color='steelblue')
ax.set_ylabel('LMP ($/MWh)')
ax.set_title('ISONE Massachusetts Hub — Day-Ahead LMP')
plt.tight_layout()
plt.show()

## 2. Preprocessing: arcsinh + MSTL

- **arcsinh** stabilizes variance while preserving the full support (including potential negatives)
- **MSTL** removes three seasonal components (24h, 168h, 8760h) and trend
- The residual r_t retains the stochastic persistence (ACF lag-1 ≈ 0.977)

In [ ]:
# arcsinh transform
lmp = df_raw['lmp'].values
arcsinh_lmp = np.arcsinh(lmp)

# MSTL decomposition
dt = pd.to_datetime(df_raw['datetime'].values)
s = pd.Series(arcsinh_lmp, index=dt)
mstl_result = MSTL(s, periods=[24, 168, 8760]).fit()

resid = mstl_result.resid.values
acf1_resid = np.corrcoef(resid[:-1], resid[1:])[0, 1]
print(f'Residual: n={len(resid)}, ACF lag-1 = {acf1_resid:.3f}')

In [ ]:
# Plot MSTL decomposition
fig, axes = plt.subplots(6, 1, figsize=(14, 10), sharex=True)
components = [
    ('arcsinh(LMP)', s.values),
    ('Trend', mstl_result.trend.values),
    ('Seasonal 24h', mstl_result.seasonal.iloc[:, 0].values),
    ('Seasonal 168h', mstl_result.seasonal.iloc[:, 1].values),
    ('Seasonal 8760h', mstl_result.seasonal.iloc[:, 2].values),
    ('Residual r_t', resid),
]
for ax, (name, vals) in zip(axes, components):
    ax.plot(dt, vals, lw=0.3, color='steelblue')
    ax.set_ylabel(name, fontsize=8)
plt.tight_layout()
plt.show()

## 3. Two Views of the Same Residual

- **r_t** (levels, persistent): input for MOMENT — the sequence model needs temporal structure
- **Δr_t** (increments, stationary): input for Feature Engineering — order-invariant stats need stationarity

In [ ]:
# Dual branch
dr = np.diff(resid)        # stationary increments
r = resid[1:]              # persistent levels (aligned with dr)
dt_aligned = dt[1:]
lmp_aligned = lmp[1:]

acf1_dr = np.corrcoef(dr[:-1], dr[1:])[0, 1]
print(f'r_t:  n={len(r)},  ACF lag-1 = {acf1_resid:.3f} (persistent)')
print(f'Δr_t: n={len(dr)}, ACF lag-1 = {acf1_dr:.3f} (stationary)')

## 4. Sliding Windows (W=512, S=6)

In [ ]:
def make_windows(vals, lmp_arr, dt_arr):
    starts = list(range(0, len(vals) - W + 1, S))
    wv = np.array([vals[s:s+W] for s in starts], dtype=np.float32)
    wl = np.array([lmp_arr[s:s+W] for s in starts], dtype=np.float32)
    ts = np.array([dt_arr[s+W-1] for s in starts])
    return wv, wl, ts

# FE receives Δr_t, MOMENT receives r_t
wr_fe, wl, ts = make_windows(dr, lmp_aligned, dt_aligned)
wr_mom, _, _  = make_windows(r, lmp_aligned, dt_aligned)
N = len(wr_fe)
print(f'Windows: N={N}, W={W}, S={S}')

## 5. Feature Engineering (15 features on Δr_t)

In [ ]:
FE_NAMES = ['mean', 'std', 'skew', 'kurt', 'min', 'max', 'range',
            'median', 'p5', 'p95', 'iqr', 'vol_24h',
            'lmp_mean', 'lmp_p95', 'lmp_std']

def compute_fe(wr, wl):
    n = len(wr)
    fe = np.empty((n, 15), dtype=np.float32)
    for i in range(n):
        x, p = wr[i].astype(np.float64), wl[i].astype(np.float64)
        fe[i, 0] = x.mean()
        fe[i, 1] = x.std()
        fe[i, 2] = float(skew(x))
        fe[i, 3] = float(kurtosis(x, fisher=False))
        fe[i, 4] = x.min()
        fe[i, 5] = x.max()
        fe[i, 6] = x.max() - x.min()
        fe[i, 7] = float(np.median(x))
        fe[i, 8] = float(np.percentile(x, 5))
        fe[i, 9] = float(np.percentile(x, 95))
        fe[i, 10] = float(np.percentile(x, 75) - np.percentile(x, 25))
        nf = (len(x) // 24) * 24
        fe[i, 11] = float(np.abs(np.diff(x[:nf].reshape(-1, 24), axis=1)).mean()) if nf >= 24 else float(np.abs(np.diff(x)).mean())
        fe[i, 12] = p.mean()
        fe[i, 13] = float(np.percentile(p, 95))
        fe[i, 14] = p.std()
    return fe

fe = compute_fe(wr_fe, wl)
print(f'FE shape: {fe.shape}')
pd.DataFrame(fe, columns=FE_NAMES).describe().round(3)

## 6. MOMENT Embedding (1024D on r_t)

MOMENT-1-large is applied **zero-shot**: no fine-tuning, no exposure to electricity data during training. It receives the persistent residual r_t and produces a 1024-dimensional embedding per window.

In [ ]:
from momentfm import MOMENTPipeline

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

model = MOMENTPipeline.from_pretrained(
    'AutonLab/MOMENT-1-large', model_kwargs={'task_name': 'embedding'})
model.init()
model = model.to(DEVICE)

BS = 64
with torch.no_grad():
    d_model = model(x_enc=torch.zeros(1, 1, W, device=DEVICE)).embeddings.shape[-1]

mom = np.empty((N, d_model), dtype=np.float32)
for s in range(0, N, BS):
    e = min(s + BS, N)
    x = torch.tensor(wr_mom[s:e], dtype=torch.float32, device=DEVICE).unsqueeze(1)
    with torch.no_grad():
        mom[s:e] = model(x_enc=x).embeddings.float().cpu().numpy()
    if (s // BS) % 20 == 0:
        print(f'  {e}/{N}')

del model, x
gc.collect()
torch.cuda.empty_cache()
print(f'MOMENT embeddings: {mom.shape}')

## 7. Diffusion Maps

Reduce each representation to a small number of diffusion coordinates. The dimensionality d is determined by the spectral gap of the eigenvalues.

In [ ]:
def diffusion_maps(X, label='', fixed_d=None):
    Xs = StandardScaler().fit_transform(X)
    if Xs.shape[1] > 50:
        Xs = PCA(n_components=50, random_state=SEED).fit_transform(Xs)

    Xt = torch.tensor(Xs, dtype=torch.float64, device=DEVICE)
    dists = torch.cdist(Xt, Xt)
    eps = float(torch.median(dists[dists > 0]).item()) ** 2
    K = torch.exp(-dists ** 2 / eps)
    P = torch.diag(1.0 / K.sum(dim=1)) @ K
    evals, evecs = torch.linalg.eigh(P)
    evals, evecs = evals.flip(0), evecs.flip(1)
    all_coords = (evecs[:, 1:21] * evals[1:21]).cpu().numpy()
    del Xt, dists, K, P
    torch.cuda.empty_cache()

    if fixed_d is not None:
        d = fixed_d
    else:
        best_d, best_s = 2, -1
        for d_cand in range(2, 21):
            cd = all_coords[:, :d_cand]
            nc = max(2, min(10, len(cd) // 50))
            km = KMeans(n_clusters=nc, n_init=5, random_state=SEED).fit(cd)
            s = silhouette_score(cd, km.labels_)
            if s > best_s:
                best_d, best_s = d_cand, s
        d = best_d

    coords = all_coords[:, :d]
    print(f'  {label}: d={d}, eigenvalues: {evals[1:d+1].cpu().numpy().round(4)}')
    return coords, d

dm_fe, d_fe = diffusion_maps(StandardScaler().fit_transform(fe), 'FE')
dm_mom, d_mom = diffusion_maps(mom, 'MOMENT', fixed_d=5)

## 8. ToMATo Clustering

ToMATo (Topological Mode Analysis Tool) identifies density modes using persistent homology. The number of clusters K emerges from the data — no a priori specification needed.

In [ ]:
from gudhi.clustering.tomato import Tomato

def tomato_cluster(X):
    best_lab, best_k, best_n = None, None, 0
    for k in [20, 40, 60, 80, 100, 150]:
        tmt = Tomato(density_type='KDE', graph_type='knn', n_neighbors=k)
        tmt.fit(X)
        if hasattr(tmt, 'diagram_') and len(tmt.diagram_) > 1:
            deaths = np.sort([d for _, d in tmt.diagram_ if d < np.inf])
            if len(deaths) > 1:
                n = len(deaths) - np.argmax(np.diff(deaths))
                tmt.n_clusters_ = n
            else:
                n = 1
        else:
            n = 1
        if n > best_n:
            best_n, best_lab, best_k = n, tmt.labels_.copy(), k
    return best_lab, best_k, best_n

lab_fe, knn_fe, modes_fe = tomato_cluster(dm_fe)
lab_mom, knn_mom, modes_mom = tomato_cluster(dm_mom)
print(f'FE:     {modes_fe} modes (k={knn_fe})')
print(f'MOMENT: {modes_mom} modes (k={knn_mom})')

## 9. Cross-Diagnostic: η² Separation Analysis

For each feature, compute η² (correlation ratio) against both partitions. This reveals which feature each branch separates best, and identifies the merge variable.

In [ ]:
def eta2(vals, labels):
    mask = labels >= 0
    x, z = vals[mask], labels[mask]
    gm = x.mean()
    ss_t = ((x - gm) ** 2).sum()
    if ss_t < 1e-15:
        return 0.0
    ss_b = sum(len(x[z == k]) * (x[z == k].mean() - gm) ** 2 for k in np.unique(z))
    return float(ss_b / ss_t)

def _acf(x, lag):
    n = len(x); m = x.mean(); v = ((x - m) ** 2).sum()
    if v < 1e-15 or lag >= n:
        return 0.0
    return float(((x[:n-lag] - m) * (x[lag:] - m)).sum() / v)

# Merge targets
lmp_mean = np.array([wl[i].mean() for i in range(N)])
acf6 = np.array([_acf(wr_mom[i].astype(np.float64), 6) for i in range(N)])

# η² for all features + ACF diagnostics
rows = []
for j, name in enumerate(FE_NAMES):
    rows.append({'feature': name,
                 'eta2_FE': round(eta2(fe[:, j], lab_fe), 3),
                 'eta2_MOM': round(eta2(fe[:, j], lab_mom), 3)})

for lag, name in [(1, 'acf_1h'), (6, 'acf_6h'), (24, 'acf_24h'), (168, 'acf_168h')]:
    vals = np.array([_acf(wr_mom[i].astype(np.float64), lag) for i in range(N)])
    rows.append({'feature': name,
                 'eta2_FE': round(eta2(vals, lab_fe), 3),
                 'eta2_MOM': round(eta2(vals, lab_mom), 3)})

df_eta = pd.DataFrame(rows)
print('Cross-diagnostic η²:')
print(df_eta.to_string(index=False))
print(f'\nFE merge variable:     lmp_mean (η²={eta2(lmp_mean, lab_fe):.3f})')
print(f'MOMENT merge variable: acf_6h   (η²={eta2(acf6, lab_mom):.3f})')

## 10. Tukey HSD Merge

Iteratively merge modes whose target variable (LMP mean for FE, ACF 6h for MOMENT) is not significantly different (Tukey HSD, α=0.05).

In [ ]:
def tukey_merge(labels, target, alpha=0.05, min_size=20):
    m = labels.copy()

    def _relabel(m):
        for j, v in enumerate(np.unique(m[m >= 0])):
            m[m == v] = j
        return m

    # Phase 1: absorb tiny modes into nearest neighbor
    u, counts = np.unique(m[m >= 0], return_counts=True)
    for k, c in sorted(zip(u, counts), key=lambda x: x[1]):
        if c < min_size and len(np.unique(m[m >= 0])) > 1:
            mk = target[m == k].mean()
            others = [kk for kk in np.unique(m[m >= 0]) if kk != k]
            nearest = min(others, key=lambda kk: abs(target[m == kk].mean() - mk))
            m[m == k] = nearest
    m = _relabel(m)

    # Phase 2: Tukey HSD merge
    while True:
        mask = m >= 0
        x, z = target[mask], m[mask]
        u = np.unique(z)
        K = len(u)
        if K <= 1:
            break
        N_total = len(x)
        g_means = np.array([x[z == k].mean() for k in u])
        g_ns = np.array([np.sum(z == k) for k in u])
        ss_within = sum(((x[z == k] - x[z == k].mean()) ** 2).sum() for k in u)
        df_within = N_total - K
        if df_within <= 0:
            break
        mse = ss_within / df_within
        order = np.argsort(g_means)
        g_means_s, g_ns_s, u_s = g_means[order], g_ns[order], u[order]
        q_crit = studentized_range.ppf(1 - alpha, K, df_within)

        best_pair, best_diff = None, np.inf
        for i in range(K):
            for j in range(i + 1, K):
                diff = abs(g_means_s[i] - g_means_s[j])
                se = np.sqrt(mse * 0.5 * (1.0 / g_ns_s[i] + 1.0 / g_ns_s[j]))
                q_stat = diff / se if se > 1e-15 else np.inf
                if q_stat < q_crit and diff < best_diff:
                    best_diff = diff
                    best_pair = (u_s[i], u_s[j])
        if best_pair is None:
            break
        m[m == best_pair[1]] = best_pair[0]
        m = _relabel(m)

    return m

lab_fe_m = tukey_merge(lab_fe, lmp_mean)
lab_mom_m = tukey_merge(lab_mom, acf6)
K_fe = len(np.unique(lab_fe_m[lab_fe_m >= 0]))
K_mom = len(np.unique(lab_mom_m[lab_mom_m >= 0]))
print(f'Economic regimes (FE):  {modes_fe} modes → {K_fe} regimes')
print(f'Dynamic regimes (MOM):  {modes_mom} modes → {K_mom} regimes')

## 11. Results: Independence Test

In [ ]:
ari = adjusted_rand_score(lab_fe_m, lab_mom_m)
print(f'ARI between economic and dynamic partitions: {ari:.3f}')
print(f'\nARI ≈ 0 confirms the two axes are statistically independent.')
print(f'Knowing the price regime tells almost nothing about the persistence regime, and vice versa.')

## 12. Regime Characterization

In [ ]:
# Economic regimes
rows_e = []
for e in np.sort(np.unique(lab_fe_m[lab_fe_m >= 0])):
    mask = lab_fe_m == e
    rows_e.append({
        'E': f'E{e}',
        'LMP mean': round(lmp_mean[mask].mean(), 1),
        'LMP std': round(lmp_mean[mask].std(), 1),
        'n': mask.sum(),
        '%': round(100 * mask.sum() / N, 1)
    })
df_E = pd.DataFrame(rows_e).sort_values('LMP mean')
print('Economic Regimes (ordered by LMP):')
print(df_E.to_string(index=False))

In [ ]:
# Dynamic regimes
rows_d = []
for d in np.sort(np.unique(lab_mom_m[lab_mom_m >= 0])):
    mask = lab_mom_m == d
    rows_d.append({
        'D': f'D{d}',
        'ACF 6h': round(acf6[mask].mean(), 3),
        'sigma_r': round(np.array([wr_mom[i].std() for i in range(N)])[mask].mean(), 3),
        'LMP mean': round(lmp_mean[mask].mean(), 1),
        'n': mask.sum(),
        '%': round(100 * mask.sum() / N, 1)
    })
df_D = pd.DataFrame(rows_d).sort_values('ACF 6h')
print('Dynamic Regimes (ordered by ACF 6h):')
print(df_D.to_string(index=False))

## 13. Visualization: Two-Axis Grid

In [ ]:
# Cross-tabulation
valid = (lab_fe_m >= 0) & (lab_mom_m >= 0)
ct = pd.crosstab(
    pd.Categorical(lab_fe_m[valid], categories=sorted(np.unique(lab_fe_m[valid]))),
    pd.Categorical(lab_mom_m[valid], categories=sorted(np.unique(lab_mom_m[valid]))),
    margins=True
)
ct.index = [f'E{i}' for i in ct.index[:-1]] + ['Total']
ct.columns = [f'D{j}' for j in ct.columns[:-1]] + ['Total']
print(f'Cross-tabulation (E x D):\n')
print(ct)
print(f'\nPopulated cells: {(ct.iloc[:-1, :-1] > 0).sum().sum()} / {(len(ct)-1) * (len(ct.columns)-1)}')

In [ ]:
# Heatmap
grid = ct.iloc[:-1, :-1].values.astype(float)
grid[grid == 0] = np.nan

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(grid, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(grid.shape[1]))
ax.set_xticklabels(ct.columns[:-1])
ax.set_yticks(range(grid.shape[0]))
ax.set_yticklabels(ct.index[:-1])
ax.set_xlabel('Dynamic regime')
ax.set_ylabel('Economic regime')
ax.set_title(f'Two-Axis Regime Grid (ARI={ari:.3f})')
plt.colorbar(im, label='Window count')

for i in range(grid.shape[0]):
    for j in range(grid.shape[1]):
        if not np.isnan(grid[i, j]):
            ax.text(j, i, f'{int(grid[i,j])}', ha='center', va='center', fontsize=7)

plt.tight_layout()
plt.show()

## 14. η² Post-Merge Confirmation

In [ ]:
eta2_fe_lmp = eta2(lmp_mean, lab_fe_m)
eta2_mom_acf = eta2(acf6, lab_mom_m)
eta2_fe_acf = eta2(acf6, lab_fe_m)
eta2_mom_lmp = eta2(lmp_mean, lab_mom_m)

print(f'η² post-merge:')
print(f'  FE  → LMP mean: {eta2_fe_lmp:.3f}   (FE separates price)')
print(f'  MOM → ACF 6h:   {eta2_mom_acf:.3f}   (MOMENT separates persistence)')
print(f'  FE  → ACF 6h:   {eta2_fe_acf:.3f}   (FE does NOT separate persistence)')
print(f'  MOM → LMP mean: {eta2_mom_lmp:.3f}   (MOMENT does NOT separate price)')
print(f'\nDiagonal pattern confirms two independent axes.')